<a href="https://colab.research.google.com/github/tarasovEgor/DeepLearningKurs/blob/main/src/lab_2/llm_transformer_lab_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Лабораторная работа 2.**

---

## LLM и Трансформер

---

## **Цель работы:** подготовить компактный корпус немецкоязычных диалогов, изучить ключевые компоненты архитектуры трансформера, выполнить fine-tuning GPT-подобной модели для генерации диалоговых ответов, сравнить оптимизаторы и scheduler, оценить качество по метрикам Perplexity, BLEU, ROUGE и протестировать модель в интерактивном режиме.


### 0. Init steps

---

#### 0.1 Установка дополнительных модулей:

In [1]:
!pip -q install transformers datasets evaluate rouge-score sentencepiece accelerate nltk spacy scikit-learn matplotlib pandas
!python -m spacy download de_core_news_sm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 43.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


#### 0.2 Импорты библиотек и базовая настройка окружения:

In [2]:
import os
import re
import math
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

import spacy
from rouge_score import rouge_scorer

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)

Устройство: cuda
